In [ ]:
import os

STATIC_WEB_PAGE = {"EXECUTE_NB", "READTHEDOCS"}.intersection(os.environ)

```{autolink-concat}
```

::::{margin}
:::{card} Chew-Mandelstam for two channels
TR-998
^^^
We visualize the branch cuts of the Chew-Mandelstam function for $S$ waves. The second sheet is reached by **taking the complex conjugate** of the Chew-Mandelstam function $\Sigma(s)$.
+++
🚧&nbsp;[compwa.github.io#204](https://github.com/ComPWA/compwa.github.io/pull/204)
:::
::::

# Chew-Mandelstam two channels

In [ ]:
%pip install -q ampform==0.14.8 sympy==1.12

In [ ]:
from __future__ import annotations

import contextlib
import inspect
import warnings
from itertools import product
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from ampform.io import aslatex
from ampform.kinematics.phasespace import Kallen
from ampform.sympy import unevaluated
from IPython.display import Math, display
from ipywidgets import FloatSlider, RadioButtons, VBox, interactive_output

warnings.filterwarnings("ignore")

In [ ]:
@unevaluated(real=False)
class PhaseSpaceFactor(sp.Expr):
    s: Any
    m1: Any
    m2: Any
    _latex_repr_ = R"\rho\left({s}\right)"

    def evaluate(self) -> sp.Expr:
        s, m1, m2 = self.args
        return 2 * sp.Abs(BreakupMomentum(s, m1, m2)) / sp.sqrt(s)


@unevaluated(real=False)
class BreakupMomentum(sp.Expr):
    s: Any
    m1: Any
    m2: Any
    _latex_repr_ = R"q\left({s}\right)"

    def evaluate(self) -> sp.Expr:
        s, m1, m2 = self.args
        return sp.sqrt(Kallen(s, m1**2, m2**2)) / (2 * sp.sqrt(s))


s, m1, m2 = sp.symbols("s m1 m2")
x, y, z = sp.symbols("x y z")
rho_expr = PhaseSpaceFactor(s, m1, m2)
q_expr = BreakupMomentum(s, m1, m2)
kallen = Kallen(x, y, z)
Math(aslatex({e: e.doit(deep=False) for e in [rho_expr, q_expr, kallen]}))

In [ ]:
@unevaluated(real=False)
class PhaseSpaceCM(sp.Expr):
    s: Any
    m1: Any
    m2: Any
    _latex_repr_ = R"\rho^\mathrm{{CM}}_{{{m1},{m2}}}\left({s}\right)"

    def evaluate(self) -> sp.Expr:
        s, m1, m2 = self.args
        return -16 * sp.pi * sp.I * ChewMandelstam(s, m1, m2)


@unevaluated(real=False)
class ChewMandelstam(sp.Expr):
    s: Any
    m1: Any
    m2: Any
    _latex_repr_ = R"\Sigma^{{{m1}}}_{{{m2}}}\left({s}\right)"

    def evaluate(self) -> sp.Expr:
        s, m1, m2 = self.args
        q = BreakupMomentum(s, m1, m2)
        return (
            1
            / (16 * sp.pi**2)
            * (
                (2 * q / sp.sqrt(s))
                * sp.log((m1**2 + m2**2 - s + 2 * sp.sqrt(s) * q) / (2 * m1 * m2))
                - (m1**2 - m2**2) * (1 / s - 1 / (m1 + m2) ** 2) * sp.log(m1 / m2)
            )
        )


rho_cm_expr = PhaseSpaceCM(s, m1, m2)
cm_expr = ChewMandelstam(s, m1, m2)
Math(aslatex({e: e.doit(deep=False) for e in [rho_cm_expr, cm_expr, q_expr]}))

### Visual comparison

In [ ]:
%matplotlib inline
%config InlineBackend.figure_formats = ['svg']
symbols = (s, m1, m2)
epsilon = 1e-10
s_values = np.linspace(-0.5, 2.5, num=1_000) + epsilon * 1j
m1_val = 0.1
m2_val = 0.7
args = (m1_val, m2_val)


fig, axes = plt.subplots(figsize=(15, 6), ncols=2, sharex=True, sharey=True)
ax1, ax2 = axes
for ax in axes:
    ax.set_xlabel("Re(s)")

rho_cm_func = sp.lambdify(symbols, rho_cm_expr.doit())
cm_values = rho_cm_func(s_values, *args)
ax1.plot(s_values, cm_values.real, label="Real part")
ax1.plot(s_values, cm_values.imag, label="Imaginary part")
ax1.set_title(f"${sp.latex(rho_cm_expr)}$")
ax1.legend()

rho_func = sp.lambdify(symbols, rho_expr.doit())
rho_values = rho_func(s_values, *args)
ax2.plot(s_values, rho_values.real, label="Real part")
ax2.plot(s_values, rho_values.imag, label="Imaginary part")
ax2.set_title(f"${sp.latex(rho_expr)}$")
ax2.legend()


ax2.set_ylim(-0.5, +1.4)
plt.tight_layout()

plt.show()

In [ ]:
%config InlineBackend.figure_formats = ['png']
fig, axes = plt.subplots(figsize=(15, 6), ncols=2, sharex=True, sharey=True)
ax1, ax2 = axes
for ax in axes:
    ax.set_xlabel(R"$\mathrm{Re}\,(s)$")
ax.set_xlabel(R"$\mathrm{Im}\,(s)$")

X, Y = np.meshgrid(
    np.linspace(s_values.real.min(), s_values.real.max(), num=500),
    np.linspace(-1, +1, num=200),
)
Z = X + Y * 1j
vmax = 1.0

ax1.pcolormesh(
    X,
    Y,
    rho_cm_func(Z, *args).real,
    cmap=plt.cm.coolwarm,
    vmin=-vmax,
    vmax=+vmax,
)
ax2.pcolormesh(
    X,
    Y,
    rho_func(Z, *args).real,
    cmap=plt.cm.coolwarm,
    vmin=-vmax,
    vmax=+vmax,
)
ax1.set_title(f"${sp.latex(rho_cm_expr)}$")
ax2.set_title(f"${sp.latex(rho_expr)}$")
fig.tight_layout()
plt.show()

## Define parameters values for plotting 

## T matrix definition with K matrix

In [ ]:
n_channels = 2
I = sp.Identity(n_channels)
K = sp.MatrixSymbol("K", n_channels, n_channels)
rho_symbol = sp.MatrixSymbol("rho", n_channels, n_channels)
rho = sp.DiagonalMatrix(rho_symbol)
rho.as_explicit()

In [ ]:
T = (I - K * rho).inv() * K
T

In [ ]:
%%time
T_expl = T.as_explicit()

In [ ]:
s = sp.Symbol("s")
m0 = sp.Symbol("m0")
w0 = sp.Symbol("Gamma0")
ma = sp.IndexedBase("m_a")
mb = sp.IndexedBase("m_b")
gamma = sp.IndexedBase("gamma")

In [ ]:
indices = tuple(range(n_channels))
k_exprs = {
    K[i, j]: (gamma[i] * gamma[j] * m0 * w0) / (s - m0**2)
    for i, j in product(indices, indices)
}
rho_exprs = {rho[i, i]: PhaseSpaceCM(s, ma[i], mb[i]) for i in indices}
subs = {**k_exprs, **rho_exprs}
Math(aslatex(subs))

In [ ]:
T_expr = sp.Matrix([[T_expl[i, j].xreplace(subs) for i in indices] for j in indices])
T_expr[0, 0].simplify(doit=False)

In [ ]:
symbols = [s, m0, w0]
symbols.extend(ma[i] for i in indices)
symbols.extend(mb[i] for i in indices)
symbols.extend(gamma[i] for i in indices)
symbols = tuple(symbols)
sp.Tuple(*symbols)

In [ ]:
T_func = np.array(
    [[sp.lambdify(symbols, T_expr[i, j].doit()) for i in indices] for j in indices]
)
T_func[0, 0]

In [ ]:
def to_slider(symbol: sp.Symbol) -> FloatSlider:
    name = str(symbol)
    _min = 0
    _max = 2
    value = 1
    if name == "m0":
        value = 2.0
        _max = 8
    if "gamma" in name:
        _min = -1
    if "Gamma" in name:
        value = 0.5
    if "_a" in name or "_b" in name:
        idx = int(name[4:-1])
        value = idx + 0.3
        if "_a" in name:
            value -= 0.1
    return FloatSlider(
        description=name,
        min=_min,
        max=_max,
        value=value,
    )


arg_names = tuple(inspect.signature(T_func[0, 0]).parameters)
sliders = {name: to_slider(s) for name, s in zip(arg_names[1:], symbols[1:])}
sliders["Tmax"] = FloatSlider(
    description="Tmax",
    min=0.01,
    max=1,
    value=0.5,
    step=0.01,
)
sliders["projection"] = RadioButtons(
    description="projection", options=["imag", "real"]
)
sliders["i"] = RadioButtons(description="i", options=indices)
sliders["j"] = RadioButtons(description="j", options=indices)

In [ ]:
%matplotlib widget
X, Y = np.meshgrid(
    np.linspace(0, 10, num=200),
    np.linspace(-2, +2, num=100),
)
Z = X + Y * 1j

epsilon = 1e-7
x = np.linspace(0, 10, num=1_000)
xn = x - epsilon * 1j
xp = x + epsilon * 1j


def plot(*, i, j, Tmax, projection, **kwargs):
    global DATA
    args = tuple(kwargs.values())
    thr = [(args[2 + i] + args[2 + i + n_channels]) ** 2 for i in indices]
    project = getattr(np, projection)
    with contextlib.suppress(ZeroDivisionError):
        T = project(T_func[i, j](Z, *args))
        Tn = project(T_func[i, j](xn, *args))
        Tp = project(T_func[i, j](xp, *args))
        Td = Tp - Tn
        if DATA is None:
            DATA = {
                "neg": ax1.plot(x, Tn, label=f"$T_{{{i},{j}}}(s+0i)$")[0],
                "pos": ax1.plot(x, Tp, label=f"$T_{{{i},{j}}}(s+0i)$")[0],
                "diff": ax1.plot(x, Td, label="Disc(s)")[0],
                "mesh": ax2.pcolormesh(X, Y, T, cmap=plt.cm.coolwarm),
            }
            for i, threshold in enumerate(thr):
                label = f"thr{i}"
                DATA[label] = ax2.axvline(threshold, c=f"C{i}", label=label)
            ax1.legend()
            ax1.legend()
        else:
            DATA["mesh"].set_array(T)
            DATA["neg"].set_ydata(Tn)
            DATA["pos"].set_ydata(Tp)
            DATA["diff"].set_ydata(Td)
            for i, threshold in enumerate(thr):
                DATA[f"thr{i}"].set_xdata(threshold)
        ax1.set_ylim(
            min(Tn.min(), Tp.min(), Td.min()),
            max(Tn.max(), Tp.max(), Td.max()),
        )
    DATA["mesh"].set_clim(-Tmax, +Tmax)
    fig.canvas.draw()


DATA = None
fig, axs = plt.subplots(figsize=(8, 8), nrows=2, sharex=True)
ax1, ax2 = axs
for ax in axs:
    ax.set_xlim(X.min(), X.max())
fig.tight_layout()
output = interactive_output(plot, controls=sliders)
ui = VBox(tuple(sliders.values()))
display(output, ui)